# V1. MLP with BatchNorm

The first model is an MLP. This model is rather simple and implemented without classes so that I can see the internal mechanics. 

In [1]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

data = torch.load('./data/dante_preprocessed.pt')


In [2]:
X_train = data["X_train"]
Y_train = data["Y_train"]

X_val = data["X_val"]
Y_val = data["Y_val"]

X_test = data["X_test"]
Y_test = data["Y_test"]

stoi = data["stoi"]
itos = data["itos"]

block_size = data["block_size"]

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("block_size:", block_size)
print("vocab_size:", len(stoi))

X_train: torch.Size([89605, 5])
Y_train: torch.Size([89605])
block_size: 5
vocab_size: 12002


In [3]:
print(X_train.shape)
print(Y_train[0])

print([itos[i.item()] for i in X_train[0]])
print(itos[Y_train[0].item()])

torch.Size([89605, 5])
tensor(338)
['<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']
al


Given the previous 5 tokens, predict what token comes next.This is why your X_train and Y_train are so important.

X = what we show the model.

Y = what we want the model to predict.

Xbatch
shape = [64, 5]

       ↓ MLP

predictions
shape = [64, 12002]  # number of possible tokens

       ↓ compare with

Ybatch
shape = [64]

---
### Initialising parameters

In [4]:
emb_dim = 384 # standard value for small models
vocab_size = len(stoi) # vocabulary of Dante
hidden_dim = 100 # initial run with 100 neurons
g = torch.Generator().manual_seed(42)

# Parameters

C = torch.randn((vocab_size, emb_dim), generator=g) # embedding matrix

W1 = torch.randn((block_size * emb_dim, hidden_dim), generator=g) * 0.1
#b1 = torch.randn(hidden_dim, generator=g) * 0.1  this is not used as batchnorm changes it

W2 = torch.randn((hidden_dim, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0 # to ensure 0 bias at the first iteration

#BatchNorm parameters
gamma = torch.ones(hidden_dim)
beta = torch.zeros(hidden_dim)
running_mean = torch.zeros(1, hidden_dim)
running_std = torch.ones(1, hidden_dim)
# zeros and ones since mean is 0 and std 1



parameters = [C, W1, W2, b2, gamma, beta]
for p in parameters:
    p.requires_grad = True # to track the parameters
print(f'Total parameters:', sum(p.nelement() for p in parameters))


Total parameters: 6013170


Inside the layer, each neuron takes the input values, multiplies them by its learned weights \(W\), adds a learned bias \(b\), and then passes the result through an activation function. In matrix form, this is \(z = xW + b\). We transpose \(W\) when the way we arrange our data requires it: the transpose doesn't change the mathematical idea, it simply makes the dimensions line up correctly for matrix multiplication. The weights determine how strongly each input contributes, while the bias shifts the neuron's output independently of the inputs. During training, backpropagation adjusts both \(W\) and \(b\) so that the network gradually produces better predictions.

Remember the forward pass:

$$ X \rightarrow C \rightarrow W_1 \rightarrow \tanh \rightarrow W_2 \rightarrow logits $$

The problem is that you have very large matrices.

For example:

$$ embcat = [32,1920] $$

and

$$ W_1 = [1920,100] $$

So each hidden neuron calculates roughly:

$$ hpreact_j = x_1w_1 + x_2w_2 + \dots + x_{1920}w_{1920} $$

If both the inputs and weights are around magnitude 1, you're adding 1,920 random products together.

The resulting values can become quite large.

Then you apply:

$$ h = \tanh(hpreact) $$

And this is important:

$$ \tanh(0)=0 $$

but

$$ \tanh(5)\approx1 $$

and

$$ \tanh(-5)\approx-1 $$

So if hpreact becomes large, tanh saturates. And then W2 was also initialized with values around 1.

So:

$$ logits = hW_2+b_2 $$

can also become enormous.

In [5]:
batch_size = 32
max_steps = 10000

for step in range(max_steps):

    # minibatch construct
    ix =torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    X_batch, Y_batch = X_train[ix], Y_train[ix]


    #forward pass 
    emb = C[X_batch]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 # + b1 

    #BatchNorm block
    hpr_mean = hpreact.mean(0, keepdim=True)
    hpr_std = hpreact.std(0, keepdim=True)

    momentum = 0.01

    running_mean = (1 - momentum) * running_mean + momentum * hpr_mean
    running_std = (1 - momentum) * running_std + momentum * hpr_std

    hpreact = gamma * (hpreact - hpr_mean)/ hpr_std + beta

    # Activation layer (hidden)
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_batch)

    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward() 

    # Update 
    lr = 0.1 
    for p in parameters:
        p.data += -lr * p.grad  # gradient descent


    if step % 1000 == 0:
        print(f'{step:5d}/{max_steps:5d}: loss:{loss.item():.5f}')


    0/10000: loss:9.59100
 1000/10000: loss:6.56666
 2000/10000: loss:6.40979
 3000/10000: loss:5.13681
 4000/10000: loss:6.26075
 5000/10000: loss:6.33364
 6000/10000: loss:5.80924
 7000/10000: loss:5.32251
 8000/10000: loss:5.86351
 9000/10000: loss:5.49606


If the model were essentially guessing uniformly among all 12,002 tokens, the cross-entropy would be approximately:

log(vocab_size)≈ 9.39

In [6]:
@torch.no_grad()

def split_loss(split):
    x, y = {
        'train': (X_train, Y_train),
        'val': (X_val, Y_val),
        'test': (X_test, Y_test),
    }[split]

    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)

    hpreact = embcat @ W1 #+ b1

    # BatchNorm using running statistics
    hpreact = gamma * (hpreact - running_mean) / running_std + beta

    h = torch.tanh(hpreact)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, y)

    print(split, loss.item())

split_loss('train')
split_loss('val')
split_loss('test')

train 5.860442161560059
val 6.314627170562744
test 6.336002826690674


--- 
Testing

In [7]:
g_sample = torch.Generator().manual_seed(12)

for _ in range(10):
    out = []
    context = [stoi['<PAD>']] * block_size
    for _ in range(10):  
        emb = C[torch.tensor([context])]
        hpreact = emb.view(1, -1) @ W1

        # applying the same running-stat BatchNorm used in split_loss,
        # otherwise W2 sees inputs on a different scale than it was trained on
        hpreact = gamma * (hpreact - running_mean) / running_std + beta
        h = torch.tanh(hpreact)

        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
        context = context[1:] + [ix]
        out.append(ix)

    print(' '.join(itos[i] for i in out))

, e la la s'oblia e come tuo nol ;
d non viso di ch'elli assessin , ch'attende se occùpi
, colui che 'l ricordivi in bevesti , cui stelle
tacerci onde crescer zanzara a sé e la ch'andavamo fin
rauni che meco 'l maestro duca ; ripigneva fanno l'albero
de la quando che ne voi fiera odio nel suo
era fu e poi questa pare il letizia la discesa
tutti , come spanda dallato a angel passo i piglia
a cima di o sua terra ; e perché ti
e come mentire erti sanza pianse divorarlo , ch'i ch'i
